In [ ]:
import numpy as np
from typing import *

# NF4 的 16 个量化级别 (QLoRA 论文)
# 按标准正态分布的分位点分配, 0 附近密、两端疏
NF4_LEVELS = np.array([-1.0, -0.6962, -0.5251, -0.3949, -0.2937, -0.1848, -0.0911, 0.0, 0.0796, 0.1609, 0.2461, 0.3379, 0.4408, 0.5626, 0.7230, 1.0])

def quantize_nf4(data):
    # 用 absmax 缩放到 [-1, 1], 然后查最近邻的 NF4 级别
    am = np.max(np.abs(data)) + 1e-12
    norm = data / am
    # argmin(|x - level|) 找到最近的量化级别索引
    idx = np.argmin(np.abs(norm.ravel()[:, None] - NF4_LEVELS[None, :]), axis=1)
    return idx.astype(np.uint8).reshape(data.shape), np.array([am])

def dequantize_nf4(q, am):
    return NF4_LEVELS[q] * am  # 查表 + 恢复 scale

def pack_4bit(m):
    # 两个 Int4 值拼成一个 byte: 偶数位存低 4 位, 奇数位存高 4 位
    M, N = m.shape
    if N % 2: m = np.pad(m, ((0, 0), (0, 1)))
    return (m[:, 1::2].astype(np.uint8) << 4) | (m[:, 0::2].astype(np.uint8) & 0x0F)

def unpack_4bit(p, shape):
    M, N = shape
    lo = p[:, :N//2].astype(np.int8) & 0x0F
    hi = (p[:, :N//2].astype(np.int8) >> 4) & 0x0F
    r = np.empty((M, N), dtype=np.int8)
    r[:, 0::2] = lo; r[:, 1::2] = hi
    return np.where(r >= 8, r - 16, r)  # 符号扩展: uint4 -> Int4

def weight_equalize(W1, W2):
    # 相邻两层间做 scale 转移: 让 W1 范围缩小, W2 范围扩大
    # (W1/s) @ (s*W2) = W1@W2, 计算等价但量化误差降低
    r1 = np.max(np.abs(W1), axis=0); r2 = np.max(np.abs(W2), axis=1)
    s = np.clip(np.sqrt(r1 / (r2 + 1e-12)), 0.1, 10.)
    return W1 / s[None, :], W2 * s[:, None]

def quantization_error(orig, recon):
    mse = np.mean((orig-recon)**2)
    sqnr = 10*np.log10(np.mean(orig**2)/(np.mean((orig-recon)**2)+1e-12))
    return dict(MSE=mse, SQNR_dB=sqnr)


In [ ]:
class MultiCodebookQuant:
    def __init__(self, K=2, N=256, d=8, seed=42):
        rng = np.random.RandomState(seed)
        self.K, self.N, self.d = K, N, d
        self.cb = rng.randn(K, N, d) * 0.1

    def quantize(self, W):
        # 每个 group 用 K 个码本向量的和来近似
        oc, ic = W.shape; ng = ic // self.d
        g = W.reshape(oc, ng, self.d)
        self.scale = np.max(np.abs(g), axis=-1, keepdims=True) + 1e-12
        norm = g / self.scale
        self.idx = np.zeros((self.K, oc, ng), dtype=np.int32)
        for o in range(oc):
            for g2 in range(ng):
                res = norm[o, g2].copy()
                for k in range(self.K):
                    # 逐码本贪心: 对残差找最近条目
                    dist = np.sum((self.cb[k] - res[None, :])**2, axis=1)
                    b = int(np.argmin(dist))
                    self.idx[k, o, g2] = b; res -= self.cb[k, b]

    def decode(self):
        oc, ng = self.idx.shape[1:]
        dec = np.zeros((oc, ng, self.d))
        for o in range(oc):
            for g in range(ng):
                for k in range(self.K):
                    dec[o, g] += self.cb[k, self.idx[k, o, g]]
        return (dec * self.scale).reshape(oc, -1)


In [ ]:
if __name__ == '__main__':
    np.random.seed(42)

    print('1. NF4 (NormalFloat4) - non-uniform 4-bit')
    x = np.random.randn(12) * 0.3
    qn, am = quantize_nf4(x); xh_nf4 = dequantize_nf4(qn, am)
    qi, si = quantize_nbit(x, 4, True); xh_i4 = dequantize_symmetric(qi, si)
    print(f'  Uniform Int4 SQNR: {quantization_error(x, xh_i4)["SQNR_dB"]:.1f}dB')
    print(f'  NF4 SQNR: {quantization_error(x, xh_nf4)["SQNR_dB"]:.1f}dB')

    print('\n2. 4-bit Packing (2 values per byte)')
    x = np.array([3, -5, 7, -8, 0, 1, -4, 6], dtype=np.int8).reshape(1, 8)
    p = pack_4bit(x); u = unpack_4bit(p, x.shape)
    print(f'  packed: {p[0]}  correct: {np.array_equal(x, u)}')

    print('\n3. Weight Equalization')
    W1 = np.random.uniform(-10, 10, (3, 4))
    W2 = np.random.uniform(-0.5, 0.5, (4, 5))
    W1e, W2e = weight_equalize(W1, W2)
    def qerr(W): return quantization_error(W, dequantize_symmetric(*quantize_symmetric(W, 4)))["SQNR_dB"]
    print(f'  W1: {qerr(W1):.1f} -> {qerr(W1e):.1f} dB')
    print(f'  W2: {qerr(W2):.1f} -> {qerr(W2e):.1f} dB')

    print('\n4. Multi-Codebook VQ (simplified AQLM)')
    vq = MultiCodebookQuant(K=2, N=16, d=4, seed=0)
    W = np.random.randn(2, 8) * .5
    vq.quantize(W); Wh = vq.decode()
    print(f'  codebooks: {vq.K}x{vq.N}, combinations: {vq.N**vq.K}')
    print(f'  VQ SQNR: {quantization_error(W, Wh)["SQNR_dB"]:.1f}dB')

def quantize_nbit(data, bit_width=4, symmetric=True, per_channel=False, channel_dim=0):
    fn = quantize_symmetric if symmetric else quantize_asymmetric
    return fn(data, bit_width, per_channel, channel_dim)

def quantize_symmetric(data, bit_width=8, per_channel=False, channel_dim=0, eps=1e-8):
    Qp = 2**(bit_width-1)-1; Qn = -(2**(bit_width-1))
    if per_channel:
        axes = tuple(i for i in range(data.ndim) if i != channel_dim)
        scale = np.max(np.abs(data), axis=axes, keepdims=True) / Qp
    else:
        scale = np.max(np.abs(data)) / Qp
    scale = np.where(scale < eps, 1.0, scale)
    q = np.clip(np.round(data/scale), Qn, Qp).astype(np.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.astype(np.float32) * scale
